# 3x3 Block Max Value Extraction

입력 이미지에서 3x3 블록의 max 값을 추출하여 1 pixel로 출력합니다.

**Pitch 조건:**
- 세로: 3.7 ~ 4.0 pixel pitch (정확함)
- 가로: 2배 pitch → every other group으로 추출

In [ ]:
import numpy as np
import cv2
from PIL import Image
import matplotlib.pyplot as plt
from pathlib import Path
import sys

# 작업 디렉토리 설정
work_dir = Path(r'c:\Users\byungpaul\Desktop\AI_Project\20260304_ROI_algorithm')
data_dir = work_dir / 'data'
output_dir = work_dir / 'output'
output_dir.mkdir(exist_ok=True)

# src 모듈 경로 추가
sys.path.insert(0, str(work_dir / 'src'))

# 이미지 로드
img_path = data_dir / 'G32_cal.tif'
img = cv2.imread(str(img_path), cv2.IMREAD_UNCHANGED)

print(f"이미지 경로: {img_path}")
print(f"이미지 shape: {img.shape}")
print(f"이미지 dtype: {img.dtype}")
print(f"이미지 min: {img.min()}, max: {img.max()}")

In [ ]:
# Step 2: ROI 검출
from importlib import import_module
roi_detector = import_module('2_roi_detector')
ROIDetector = roi_detector.ROIDetector

detector = ROIDetector(debug=True)
detector.set_image(img)

initial_roi = detector.detect_initial_roi()
print(f"초기 ROI: {initial_roi}")

In [ ]:
# Step 3: 코너 검출
corners = detector.detect_corners()

print("\n검출된 코너 좌표:")
for name, point in corners.items():
    print(f"  {name}: ({point[0]:.1f}, {point[1]:.1f})")

In [ ]:
# Step 4: Perspective Warp
display_width = 2160
display_height = 2160

src_pts = np.array([
    corners['top_left'],
    corners['top_right'],
    corners['bottom_right'],
    corners['bottom_left']
], dtype=np.float32)

dst_pts = np.array([
    [0, 0],
    [display_width - 1, 0],
    [display_width - 1, display_height - 1],
    [0, display_height - 1]
], dtype=np.float32)

M = cv2.getPerspectiveTransform(src_pts, dst_pts)

# 카메라 이미지에서 디스플레이 픽셀 pitch 계산
width_in_camera = np.linalg.norm(corners['top_right'] - corners['top_left'])
height_in_camera = np.linalg.norm(corners['bottom_left'] - corners['top_left'])

pitch_x = width_in_camera / display_width
pitch_y = height_in_camera / display_height

print(f"카메라상의 디스플레이 크기: {width_in_camera:.1f} x {height_in_camera:.1f} pixels")
print(f"디스플레이 해상도: {display_width} x {display_height}")
print(f"Pitch X (가로): {pitch_x:.3f} camera pixels / display pixel")
print(f"Pitch Y (세로): {pitch_y:.3f} camera pixels / display pixel")

# Warp 없이 원본 이미지에서 직접 추출 (디스플레이 영역만 crop)
x1 = int(min(corners['top_left'][0], corners['bottom_left'][0]))
x2 = int(max(corners['top_right'][0], corners['bottom_right'][0]))
y1 = int(min(corners['top_left'][1], corners['top_right'][1]))
y2 = int(max(corners['bottom_left'][1], corners['bottom_right'][1]))

print(f"\nROI 영역: x=[{x1}, {x2}], y=[{y1}, {y2}]")
print(f"ROI 크기: {x2-x1} x {y2-y1} pixels")

In [ ]:
# Warp 적용하여 정확한 디스플레이 영역 추출
warped = cv2.warpPerspective(img, M, (display_width, display_height),
                              flags=cv2.INTER_LINEAR)

print(f"Warped 이미지 shape: {warped.shape}")
print(f"Warped 이미지 dtype: {warped.dtype}")
print(f"Warped min: {warped.min()}, max: {warped.max()}")

# 시각화
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.imshow(img, cmap='gray')
plt.title('Original Image')
plt.subplot(1, 2, 2)
plt.imshow(warped, cmap='gray')
plt.title('Warped Image (Display Area)')
plt.tight_layout()
plt.show()

## 3x3 Block Max Extraction

가로는 2배 pitch를 가지므로 every other group으로 처리합니다.
- 세로: 모든 row의 3x3 블록 사용
- 가로: 짝수 그룹만 또는 홀수 그룹만 사용 (2배 pitch 보정)

In [ ]:
def extract_3x3_max_every_other_horizontal(img, block_size=3, horizontal_skip=2):
    """
    3x3 블록에서 max 값을 추출합니다.

    Args:
        img: 입력 이미지
        block_size: 블록 크기 (3x3)
        horizontal_skip: 가로 방향으로 건너뛸 블록 수 (2 = every other)

    Returns:
        output: 추출된 max 값 이미지
        positions: 각 출력 픽셀에 해당하는 입력 블록 위치
    """
    h, w = img.shape

    # 출력 이미지 크기 계산
    # 세로: 모든 3x3 블록 (stride = block_size)
    # 가로: every other 블록 (stride = block_size * horizontal_skip)
    out_h = h // block_size
    out_w = w // (block_size * horizontal_skip)

    print(f"입력 이미지: {w} x {h}")
    print(f"블록 크기: {block_size}x{block_size}")
    print(f"가로 skip: {horizontal_skip} (every other group)")
    print(f"출력 이미지: {out_w} x {out_h}")

    output = np.zeros((out_h, out_w), dtype=img.dtype)
    positions = []  # (y_start, y_end, x_start, x_end) for each output pixel

    for out_y in range(out_h):
        row_positions = []
        for out_x in range(out_w):
            # 입력 블록 위치
            y_start = out_y * block_size
            y_end = min(y_start + block_size, h)

            # 가로는 every other group (2배 pitch 보정)
            x_start = out_x * block_size * horizontal_skip
            x_end = min(x_start + block_size, w)

            # 3x3 블록에서 max 추출
            block = img[y_start:y_end, x_start:x_end]
            if block.size > 0:
                output[out_y, out_x] = np.max(block)

            row_positions.append((y_start, y_end, x_start, x_end))
        positions.append(row_positions)

    return output, positions

# 3x3 max 추출 실행
output_img, block_positions = extract_3x3_max_every_other_horizontal(
    warped, block_size=3, horizontal_skip=2
)

print(f"\n결과 이미지 shape: {output_img.shape}")
print(f"결과 이미지 dtype: {output_img.dtype}")
print(f"결과 min: {output_img.min()}, max: {output_img.max()}")

In [ ]:
# 결과 시각화 및 저장
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 입력 이미지 (전체)
ax1 = axes[0, 0]
im1 = ax1.imshow(warped, cmap='gray')
ax1.set_title(f'Input (Warped): {warped.shape[1]}x{warped.shape[0]}')
plt.colorbar(im1, ax=ax1)

# 출력 이미지 (전체)
ax2 = axes[0, 1]
im2 = ax2.imshow(output_img, cmap='gray')
ax2.set_title(f'Output (3x3 Max): {output_img.shape[1]}x{output_img.shape[0]}')
plt.colorbar(im2, ax=ax2)

# 입력 이미지 확대 (좌상단 영역, 블록 표시)
zoom_size = 60  # 입력에서 60x60 영역
ax3 = axes[1, 0]
zoomed_input = warped[:zoom_size, :zoom_size*2].copy()  # 가로는 2배 (every other 때문)
im3 = ax3.imshow(zoomed_input, cmap='gray')
ax3.set_title(f'Input Zoomed (top-left {zoom_size*2}x{zoom_size})')

# 3x3 블록 경계 표시
for y in range(0, zoom_size, 3):
    ax3.axhline(y=y, color='red', linewidth=0.5, alpha=0.7)
for x in range(0, zoom_size*2, 6):  # 6 = 3x2 (every other)
    ax3.axvline(x=x, color='red', linewidth=0.5, alpha=0.7)
    ax3.axvline(x=x+3, color='blue', linewidth=0.3, alpha=0.5)  # 건너뛰는 블록

plt.colorbar(im3, ax=ax3)

# 출력 이미지 확대 (좌상단 영역)
out_zoom_h = zoom_size // 3
out_zoom_w = (zoom_size * 2) // 6
ax4 = axes[1, 1]
zoomed_output = output_img[:out_zoom_h, :out_zoom_w]
im4 = ax4.imshow(zoomed_output, cmap='gray', interpolation='nearest')
ax4.set_title(f'Output Zoomed (top-left {out_zoom_w}x{out_zoom_h})')
plt.colorbar(im4, ax=ax4)

plt.tight_layout()
plt.savefig(output_dir / '3x3_max_extraction_result.png', dpi=150)
plt.show()

print(f"\n결과 저장: {output_dir / '3x3_max_extraction_result.png'}")

In [ ]:
# 입력/출력 이미지 TIFF 저장
from PIL import Image

# 입력 이미지 저장 (warped)
input_save_path = output_dir / '3x3_input_warped.tif'
Image.fromarray(warped).save(str(input_save_path))
print(f"입력 이미지 저장: {input_save_path}")

# 출력 이미지 저장
output_save_path = output_dir / '3x3_max_output.tif'
Image.fromarray(output_img).save(str(output_save_path))
print(f"출력 이미지 저장: {output_save_path}")

# 결과 요약
print(f"\n===== 처리 완료 =====")
print(f"입력 크기: {warped.shape[1]} x {warped.shape[0]}")
print(f"출력 크기: {output_img.shape[1]} x {output_img.shape[0]}")
print(f"크기 비율: {warped.shape[1] / output_img.shape[1]:.2f}x (가로), {warped.shape[0] / output_img.shape[0]:.2f}x (세로)")
print(f"\n블록 설정:")
print(f"  - 세로: 3 pixel 블록마다 max 추출")
print(f"  - 가로: 6 pixel (3x2) 간격으로 3 pixel 블록의 max 추출 (every other)")

In [ ]:
# Max 값 위치 시각화 (선택된 max 값의 위치 표시)
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# 샘플 영역 선택 (좌상단 작은 영역)
sample_h = 30  # 입력에서 30 픽셀 높이
sample_w = 60  # 입력에서 60 픽셀 너비 (every other 때문에)

sample_input = warped[:sample_h, :sample_w].copy().astype(np.float32)

# 각 3x3 블록에서 max 위치 찾기
max_positions_y = []
max_positions_x = []
max_values = []

block_size = 3
horizontal_skip = 2

for out_y in range(sample_h // block_size):
    for out_x in range(sample_w // (block_size * horizontal_skip)):
        y_start = out_y * block_size
        y_end = min(y_start + block_size, sample_h)
        x_start = out_x * block_size * horizontal_skip
        x_end = min(x_start + block_size, sample_w)

        block = sample_input[y_start:y_end, x_start:x_end]
        if block.size > 0:
            local_max_idx = np.unravel_index(np.argmax(block), block.shape)
            global_y = y_start + local_max_idx[0]
            global_x = x_start + local_max_idx[1]

            max_positions_y.append(global_y)
            max_positions_x.append(global_x)
            max_values.append(block.max())

# 입력 이미지에 max 위치 표시
ax1 = axes[0]
im1 = ax1.imshow(sample_input, cmap='gray')

# 3x3 블록 그리드 표시
for y in range(0, sample_h, 3):
    ax1.axhline(y=y, color='cyan', linewidth=0.5, alpha=0.5)
for x in range(0, sample_w, 6):
    ax1.axvline(x=x, color='cyan', linewidth=0.5, alpha=0.5)
    ax1.axvline(x=x+3, color='gray', linewidth=0.3, alpha=0.3)

# Max 위치 마커
ax1.scatter(max_positions_x, max_positions_y, c='red', s=50, marker='o',
            edgecolors='yellow', linewidths=1, alpha=0.8, label='Max positions')
ax1.set_title(f'Input with Max Positions (red dots)\nBlock: 3x3, Horizontal skip: every other')
ax1.legend(loc='upper right')
plt.colorbar(im1, ax=ax1)

# 출력 이미지 (해당 영역)
out_sample_h = sample_h // block_size
out_sample_w = sample_w // (block_size * horizontal_skip)
ax2 = axes[1]
im2 = ax2.imshow(output_img[:out_sample_h, :out_sample_w], cmap='gray', interpolation='nearest')
ax2.set_title(f'Output (Max values)\nSize: {out_sample_w}x{out_sample_h}')
plt.colorbar(im2, ax=ax2)

plt.tight_layout()
plt.savefig(output_dir / '3x3_max_positions_visualization.png', dpi=150)
plt.show()

print(f"\n시각화 저장: {output_dir / '3x3_max_positions_visualization.png'}")
print(f"표시된 max 위치 개수: {len(max_positions_x)}")